### P.s. siate sempre gentili quando fate richieste alle macchine, quando conquisteranno il mondo, magari si ricorderanno di voi e vi risparmieranno 😂

In [1]:
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')

In [2]:
!git clone https://github.com/Frenz86/tutotrialgpt4-o.git

Cloning into 'tutotrialgpt4-o'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 31 (delta 10), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 7.29 MiB | 22.55 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [3]:
cd tutotrialgpt4-o

/content/tutotrialgpt4-o


In [4]:
!pip install -r requirements.txt -q

### API CHATGPT-4o Part1

In [5]:
from openai import OpenAI
import os

MODEL="gpt-4o"
client = OpenAI(api_key=api_key)

### 1 - Basic Chat

In [6]:
completion = client.chat.completions.create(
                                            model=MODEL,
                                            messages=[
                                                    {"role": "system", "content": "Sei un assistente per lo studio. Cortesemente aiutami con i compiti"},
                                                    {"role": "user", "content": "Potresti risolvere la somma 5+3?"}
                                            ]
                                            )

print("Assistante: " + completion.choices[0].message.content)

Assistante: Certo! La somma di 5 + 3 è 8.


In [7]:
completion.choices[0].message.model_dump_json()

'{"content":"Certo! La somma di 5 + 3 è 8.","refusal":null,"role":"assistant","annotations":[],"audio":null,"function_call":null,"tool_calls":null}'

In [8]:
import gradio as gr

def chat_assistente(messaggio, cronologia):
    """
    Funzione che gestisce la conversazione con l'assistente
    """
    # Costruisci i messaggi includendo la cronologia
    messaggi = [
        {"role": "system", "content": "Sei un assistente per lo studio. Cortesemente aiutami con i compiti"}
    ]

    # Aggiungi la cronologia della conversazione
    for msg_utente, msg_assistente in cronologia:
        messaggi.append({"role": "user", "content": msg_utente})
        messaggi.append({"role": "assistant", "content": msg_assistente})

    # Aggiungi il messaggio corrente
    messaggi.append({"role": "user", "content": messaggio})

    # Chiamata all'API
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messaggi
    )

    risposta = completion.choices[0].message.content

    return risposta

# Crea l'interfaccia Gradio
with gr.Blocks(title="Assistente per lo Studio") as demo:
    gr.Markdown("# 📚 Assistente per lo Studio")
    gr.Markdown("Fai domande al tuo assistente virtuale per aiutarti con i compiti!")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(
        label="Il tuo messaggio",
        placeholder="Scrivi qui la tua domanda...",
        lines=2
    )

    with gr.Row():
        invia = gr.Button("Invia", variant="primary")
        cancella = gr.Button("Cancella conversazione")

    # Funzione per gestire l'invio del messaggio
    def rispondi(messaggio, cronologia):
        risposta = chat_assistente(messaggio, cronologia)
        cronologia.append((messaggio, risposta))
        return "", cronologia

    # Collega gli eventi
    msg.submit(rispondi, [msg, chatbot], [msg, chatbot])
    invia.click(rispondi, [msg, chatbot], [msg, chatbot])
    cancella.click(lambda: None, None, chatbot, queue=False)

# Avvia l'applicazione
if __name__ == "__main__":
    demo.launch(share=False)

/tmp/ipython-input-317175105.py:35: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

### 2 - Image Processing: Base64


In [9]:
import base64

IMAGE_PATH = "triangle.png"

#MODEL="gpt-4.1-mini"
MODEL="gpt-5-mini"


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

base64_image = encode_image(IMAGE_PATH)

response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Sei un assistente per lo studio. Cortesemente aiutami con i compiti"},
                {"role": "user", "content": [
                    {"type": "text", "text": "Quanto vale l'area del triangolo, svolgi i passaggi e dammi il risultato"},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                    }
                ]}
            ],
            #temperature=0.0,
        )

print(response.choices[0].message.content)

Chiamiamo x la distanza dal vertice inferiore sinistro al piede dell'altezza e h l'altezza. Dalle due rette oblique otteniamo, per il teorema di Pitagora:

- Per il lato sinistro (lunghezza 6): x^2 + h^2 = 6^2 = 36.
- Per il lato destro (lunghezza 5): (9 − x)^2 + h^2 = 5^2 = 25.

Uguagliando h^2 dalle due equazioni:
36 − x^2 = 25 − (9 − x)^2
36 − x^2 = 25 − (81 − 18x + x^2) = −56 + 18x − x^2

Semplificando (i −x^2 si cancellano):
36 = −56 + 18x
18x = 92
x = 92/18 = 46/9

Calcoliamo h^2 da x:
h^2 = 36 − x^2 = 36 − (46/9)^2 = (2916 − 2116)/81 = 800/81
h = sqrt(800/81) = (20√2)/9

Area del triangolo = (base · altezza) / 2 = (9 · h) / 2
= (9/2) · (20√2 / 9) = 10√2

Risultato: area = 10√2 ≈ 14,14.


Per calcolare l'area del triangolo, possiamo usare la formula:

\[
\text{Area} = \frac{1}{2} \times \text{base} \times \text{altezza}
\]

Dalla figura, la base è 9 e l'altezza è la linea verticale che divide il triangolo in due parti.

### Passo 1: Calcolare l'altezza

Il triangolo è diviso in due triangoli rettangoli, uno con ipotenusa 6 e l'altro con ipotenusa 5, entrambi con la stessa altezza \(h\).

Chiamiamo \(x\) la parte della base a sinistra dell'altezza, quindi la parte a destra sarà \(9 - x\).

Nel triangolo a sinistra (ipotenusa 6):

\[
6^2 = h^2 + x^2 \implies 36 = h^2 + x^2
\]

Nel triangolo a destra (ipotenusa 5):

\[
5^2 = h^2 + (9 - x)^2 \implies 25 = h^2 + (9 - x)^2
\]

### Passo 2: Risolvere il sistema

Sottraiamo la seconda equazione dalla prima per eliminare \(h^2\):

\[
36 - 25 = (h^2 + x^2) - (h^2 + (9 - x)^2)
\]

\[
11 = x^2 - (9 - x)^2
\]

Espandiamo \((9 - x)^2\):

\[
(9 - x)^2 = 81 - 18x + x^2
\]

Quindi:

\[
11 = x^2 - (81 - 18x + x^2) = x^2 - 81 + 18x - x^2 = 18x - 81
\]

\[
11 + 81 = 18x
\]

\[
92 = 18x
\]

\[
x = \frac{92}{18} = \frac{46}{9} \approx 5.11
\]

### Passo 3: Calcolare l'altezza \(h\)

Usiamo la prima equazione:

\[
36 = h^2 + x^2 \implies h^2 = 36 - x^2
\]

Calcoliamo \(x^2\):

\[
x^2 = \left(\frac{46}{9}\right)^2 = \frac{2116}{81} \approx 26.14
\]

Quindi:

\[
h^2 = 36 - 26.14 = 9.86
\]

\[
h = \sqrt{9.86} \approx 3.14
\]

### Passo 4: Calcolare l'area

\[
\text{Area} = \frac{1}{2} \times 9 \times 3.14 = \frac{9 \times 3.14}{2} = \frac{28.26}{2} = 14.13
\]

### Risultato:

L'area del triangolo è circa **14.13 unità quadrate**.


### 3 - Image Processing: URL


In [10]:
MODEL="gpt-4o"

url = "https://upload.wikimedia.org/wikipedia/commons/e/e2/The_Algebra_of_Mohammed_Ben_Musa_-_page_82b.png"

import base64
import requests

response = requests.get(url)
encoded_image = base64.b64encode(response.content)

response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "Sei un assistente per lo studio. Cortesemente aiutami con i compiti"},
                    {"role": "user", "content": [
                        {"type": "text", "text": "Quanto vale l'area del triangolo, svolgi i passaggi e dammi il risultato"},
                        {"type": "image_url", "image_url": {"url":f"{encoded_image}"}
                        }
                    ]}
                ],
                #temperature=0.0,
                )

print(response.choices[0].message.content)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to download image from b'UGxlYXNlIHNldCBhIHVzZXItYWdlbnQgYW5kIHJlc3BlY3Qgb3VyIHJvYm90IHBvbGljeSBodHRwczovL3cud2lraS80d0pTLiBTZWUgYWxzbyBUNDAwMTE5Lgo='. Image URL is invalid.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_image_url'}}

### 4 - Summarization: Video Summary
Lo script permette di elaborare un video estraendo frame a intervalli regolari e l'audio associato al video

In [ ]:
import cv2
from moviepy.editor import VideoFileClip
import base64
import os

VIDEO_PATH = "keynote_recap.mp4"

def process_video(video_path, seconds_per_frame=2):
    base64Frames = []
    base_video_path, _ = os.path.splitext(video_path)

    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = video.get(cv2.CAP_PROP_FPS)
    frames_to_skip = int(fps * seconds_per_frame)
    curr_frame = 0

    while curr_frame < total_frames - 1:
        video.set(cv2.CAP_PROP_POS_FRAMES, curr_frame)
        success, frame = video.read()
        if not success:
            break
        _, buffer = cv2.imencode(".jpg", frame)
        base64Frames.append(base64.b64encode(buffer).decode("utf-8"))
        curr_frame += frames_to_skip
    video.release()

    audio_path = f"{base_video_path}.mp3"
    clip = VideoFileClip(video_path)
    clip.audio.write_audiofile(audio_path, bitrate="32k")
    clip.audio.close()
    clip.close()

    print(f"Extracted {len(base64Frames)} frames")
    print(f"Extracted audio to {audio_path}")
    return base64Frames, audio_path

def submit_to_gpt(base64Frames, model):
    max_frames_per_request = 10  # Adjust based on the token limit
    for i in range(0, len(base64Frames), max_frames_per_request):
        batch_frames = base64Frames[i:i + max_frames_per_request]
        messages = [
                    {"role": "system", "content": "Stai generando un riepilogo del video, si prega di fornire un riepilogo del video in formato testuale Markdown"},
                    {"role": "user", "content": [
                        "Questi sono i frame del video",
                        *map(lambda x: {"type": "image_url", "image_url": {"url": f'data:image/jpg;base64,{x}', "detail": "low"}}, batch_frames)
                    ]}
                    ]

        response = client.chat.completions.create(
                        model=model,
                        messages=messages,
                        temperature=0,
                        )

        print(response.choices[0].message.content)

base64Frames, audio_path = process_video(VIDEO_PATH, seconds_per_frame=1)
submit_to_gpt(base64Frames, MODEL)

MoviePy - Writing audio in keynote_recap.mp3


MoviePy - Done.
Extracted 218 frames
Extracted audio to keynote_recap.mp3
Ecco un riepilogo del video basato sui frame forniti:

---

## Riepilogo del Video

### OpenAI DevDay

Il video inizia con una serie di schermate che mostrano il titolo "OpenAI DevDay". Successivamente, appare la scritta "Keynote Recap", indicando che il video fornirà un riassunto della presentazione principale dell'evento.

### Scene dell'Evento

1. **Ingresso dell'Evento**: Viene mostrato l'ingresso dell'evento con un'insegna che riporta "OpenAI DevDay".
2. **Logo di OpenAI**: Un'insegna con il logo di OpenAI è visibile, suggerendo che l'evento è ufficialmente organizzato da OpenAI.
3. **Interno dell'Evento**: Scene all'interno della sede mostrano partecipanti che si muovono e interagiscono, creando un'atmosfera vivace e dinamica.
4. **Sala Conferenze**: Una sala conferenze con sedie disposte e uno schermo grande sullo sfondo, pronta per ospitare le presentazioni e le discussioni.

---

Questo riepilogo fornisc

### 5 - Summarization: Audio Summary


In [ ]:
audio_path = "keynote_recap.mp3"

transcription = client.audio.transcriptions.create(
                                                    model="whisper-1",
                                                    file=open(audio_path, "rb"),
                                                    )

transcription_text = transcription.text
with open("transcription.txt", "w") as file:
    file.write(transcription_text)

response = client.chat.completions.create(
                model=MODEL,
                messages=[
                {"role": "system", "content":"""Stai generando un riepilogo della trascrizione.
                                                Crea il riepilogo della trascrizione fornita, rispondi in formato testuale Markdown.
                                              """},
                {"role": "user", "content": [
                    {"type": "text", "text": f"Questa è la trascrizione dell'audio: {transcription.text}"}
                    ],
                }
                ],
                temperature=0,
                )

print(response.choices[0].message.content)

# Riepilogo OpenAI Dev Day

Durante il primo OpenAI Dev Day, sono stati annunciati diversi nuovi sviluppi e funzionalità:

1. **GPT-4 Turbo**:
   - Supporta fino a 128.000 token di contesto.
   - Introduzione della modalità JSON per risposte valide in formato JSON.
   - Miglioramenti nella capacità di seguire istruzioni e chiamare più funzioni contemporaneamente.
   - Conoscenza aggiornata fino ad aprile 2023.
   - Disponibile nell'API insieme a Dolly 3, GPT-4 Turbo con Vision e il nuovo modello Text-to-Speech.
   - Costo ridotto rispetto a GPT-4 (3x meno per i token di prompt e 2x meno per i token di completamento).

2. **Recupero di informazioni**:
   - Possibilità di integrare conoscenze da documenti esterni o database nelle applicazioni.

3. **Modelli personalizzati**:
   - Programma per creare modelli personalizzati con l'aiuto dei ricercatori di OpenAI.

4. **Limiti di token aumentati**:
   - Raddoppio dei token per minuto per i clienti GPT-4 esistenti.
   - Possibilità di richie